# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tushar-sharma001/Flyrank-Ml-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 2 — Refresh / Content Opportunity Scoring.**

I'm picking this lane because it produces something a real reviewer could act on this week (a ranked queue), not just an interesting pattern. It also matches how I already think about problems — a lot of my past project work has been about turning noisy signals into a ranked/scored output someone can act on (e.g. an RFM-based customer scoring system on a separate project), so this lane plays to a way of working I already trust. The starter dataset also backs this choice with real volume, not a handful of edge cases — see Section 3.

In [22]:
import os

if not os.path.exists("Flyrank-Ml-Internship"):
    !git clone https://github.com/tushar-sharma001/Flyrank-Ml-Internship.git

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** out of thousands of pages with search visibility, which ones should
a content reviewer look at first this week?

**Who acts, and how:** a content editor/reviewer, working through a
capacity-limited queue — they can review maybe a few dozen pages a week, not
every page in the dataset. They open the ranked list, read the reason codes,
and decide to refresh, expand, protect, prune, or monitor each candidate.

**Cost of a wrong call, in both directions:**
- *False positive* — the model flags a page that didn't actually need attention.
  Cost: wasted editor hours on a page that was fine.
- *False negative* — a page that's genuinely losing visibility gets missed.
  Cost: the decline keeps compounding, unnoticed, until it's a much bigger loss.

Because editor time is the scarce resource, precision at the top of the ranked
list (precision@K) matters more here than catching every possible case (recall)
— this is a threshold/policy choice, not a fixed rule (lane guide, Section 11).

In [23]:
import pandas as pd
from pathlib import Path

# Robust path: works whether run from the repo root (Colab) or from work/notebooks (local)
candidates = [Path("data/raw/content_refresh_anonymized.csv"),
              Path("../../data/raw/content_refresh_anonymized.csv"),
              Path("Flyrank-Ml-Internship/data/raw/content_refresh_anonymized.csv")]
csv_path = next(p for p in candidates if p.exists())

df = pd.read_csv(csv_path)

# Apply the lane guide's standard starter filter before counting anything
df_f = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")

declining_with_demand = ((df_f["trend_direction"] == "down") & (df_f["impressions_90d"] >= 100)).sum()
total = len(df_f)

print(f"Rows after the standard filter: {total}")
print(f"'declining_with_demand' backlog: {declining_with_demand} pages "
      f"({100 * declining_with_demand / total:.1f}% of the filtered dataset)")
print("This is the rough size of the queue a reviewer would be working through without ranking help.")

Rows after the standard filter: 30000
'declining_with_demand' backlog: 13152 pages (43.8% of the filtered dataset)
This is the rough size of the queue a reviewer would be working through without ranking help.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Loaded the 30,000-row starter dataset (`data/raw/content_refresh_anonymized.csv`), applied the
lane guide's standard filter (`impressions_90d > 0`, `content_age_days >= 90`), and checked how
big and how real the ranking problem actually is:

- **16,262 of 30,000 pages (54.2%)** currently show `trend_direction == "down"` — more than half
  the inventory, which is far too many for a human to just eyeball in order.
- **13,152 pages (43.8% of the dataset)** are "declining with demand" — losing ground *and*
  still pulling at least 100 impressions in the last 90 days. This excludes pages nobody was
  seeing anyway, so it's a meaningful backlog, not noise.
- The **median page gets 731 impressions in 90 days**, across **32 different clients** — so
  there's real, varied volume behind this, not a handful of pages from one site.

This is enough real signal and enough volume to make ranking genuinely useful — a reviewer
manually working through 13,000+ candidates in trend-direction order alone, with no other
signal, is not a realistic plan.

In [24]:
import pandas as pd
from pathlib import Path

# Robust path: works whether run from the repo root (Colab) or from work/notebooks (local)
candidates = [Path("data/raw/content_refresh_anonymized.csv"),
              Path("../../data/raw/content_refresh_anonymized.csv"),
              Path("Flyrank-Ml-Internship/data/raw/content_refresh_anonymized.csv")]
csv_path = next(p for p in candidates if p.exists())

df = pd.read_csv(csv_path)
df_f = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].drop_duplicates("content_id")
total = len(df_f)

down_count = (df_f["trend_direction"] == "down").sum()
declining_with_demand = ((df_f["trend_direction"] == "down") & (df_f["impressions_90d"] >= 100)).sum()
median_impressions = df_f["impressions_90d"].median()
n_clients = df_f["client_id"].nunique()

print(f"Total rows after filter: {total}")
print(f"Pages with trend_direction == 'down': {down_count} ({100*down_count/total:.1f}%)")
print(f"'declining_with_demand' pages (down AND impressions_90d >= 100): "
      f"{declining_with_demand} ({100*declining_with_demand/total:.1f}%)")
print(f"Median impressions_90d: {median_impressions}")
print(f"Distinct clients represented: {n_clients}")# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Total rows after filter: 30000
Pages with trend_direction == 'down': 16262 (54.2%)
'declining_with_demand' pages (down AND impressions_90d >= 100): 13152 (43.8%)
Median impressions_90d: 731.0
Distinct clients represented: 32


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** observed and directional patterns in this snapshot — for example, "this
page is losing search visibility relative to its own recent history" or "this page sits below
where its position tier would predict for CTR." The final output is decision-support: a ranked
review queue with reason codes a human can inspect, meant to help a reviewer spend limited time
on the most promising candidates first.

**What I can't claim:** that refreshing a flagged page will *cause* it to recover — proving that
needs an actual experiment or causal design, which this data alone can't give me (lane guide,
Section 6). I also can't claim anything about how Google's algorithm actually works, and I can't
call a drop a real "decline" without first ruling out consolidation, seasonality, and plain noise
(Section 7) — a drop that looks scary in isolation might just be a sibling page absorbing the
same demand.

One more discipline check: FlyRank's own product-decision fields (`health_score`,
`priority_score`, `action_type`, and similar) are deliberately **not shipped** in this dataset,
so there's nothing here I could accidentally use as a feature that would let my model just copy
an existing rule instead of finding its own signal — confirmed below.

In [25]:
banned_product_columns = ["health_score", "priority_score", "action_type",
                          "needs_ctr_fix", "is_quick_win", "refresh_tier"]
present = [c for c in banned_product_columns if c in df.columns]

print(f"Product-decision columns present in the starter dataset: {present}")
print("Empty list confirms the dataset ships observable signals only, per the lane guide.")

Product-decision columns present in the starter dataset: []
Empty list confirms the dataset ships observable signals only, per the lane guide.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.